# Ablation Row 3 — + Multi-scale pyramid (inference only)

| | Value |
|---|---|
| Features | full (same as row 2) |
| PCA | 4677 components (same as row 2) |
| Scales | **(0.75, 1.0, 1.5)** + cross-scale NMS |
| HNM | off |
| Threshold | `cfg.detection_threshold = 1.5` (default) |

Delta vs row 2: turn on the image pyramid. **No retraining** — loads
`microglia-artifacts-row2/`'s `svm_clf` / `scaler` / `pca` and runs multi-scale
detection on the test set. Same model, same patches, only the inference-time
sweep changes.

This isolates the pyramid contribution from any classifier-level confound.

## 1  Imports + Config + load row 2 model

In [ ]:
import os
import joblib
import numpy as np
import matplotlib.image as mpimg

from pipeline import (
    Config,
    extract_raw_features, fit_pipeline,
    tune_svm, train_svm, evaluate_classifier,
    process_image, multi_scale_detect, non_max_suppression,
    load_gt_boxes, evaluate_detections, plot_gt_vs_pred,
    save_hard_negatives, tune_detection_threshold,
    ensure_test_patches, patch_level_test_eval, detection_level_test_eval,
    extract_features, list_images,
)

%matplotlib inline

In [ ]:
# Inference-only: load the row 2 model and override scale_factors at the cfg
# level. Artifact paths still point at row 2 so we don't accidentally write here.
cfg = Config(
    artifact_dir='./microglia-artifacts-row2',
    feature_mode='full',
    scale_factors=(0.75, 1.0, 1.5),
    pca_n_components=4677,
    detection_threshold=1.5,
)
print(cfg)

svm_clf = joblib.load(cfg.svm_clf_path)
scaler  = joblib.load(cfg.scaler_path)
pca     = joblib.load(cfg.pca_path)
print(f"Loaded model from {cfg.artifact_dir}/")

## 2  Test evaluation (no retraining)

In [ ]:
ensure_test_patches(cfg)

print("── Patch-level test (unchanged from row 2) ──")
patch_level_test_eval(svm_clf, scaler, pca, cfg)

print("\n── Detection-level test (multi-scale, t=%.2f) ──" % cfg.detection_threshold)
row3_metrics = detection_level_test_eval(svm_clf, scaler, pca, cfg, show_plots=True)